In [1]:
import requests
import time
import pandas as pd
from parsel import Selector
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# Setup Selenium WebDriver
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode (no browser UI)
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--window-size=1920x1080")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

# List of category URLs
office_furniture_urls = [
    "https://store.hermanmiller.com/office/constant/task-chairs?lang=en_US",
    "https://store.hermanmiller.com/office/constant/home-office-storage-hm?lang=en_US",
    "https://store.hermanmiller.com/office/constant/dining-tables?lang=en_US",
    "https://store.hermanmiller.com/office/constant/sit-to-stand-desks?lang=en_US",
    "https://store.hermanmiller.com/office-view-all/constant/desk-accessories-hm?lang=en_US"
]

base_url = "https://store.hermanmiller.com"
full_links = []

# ✅ Step 1: Fetch product links from all categories (Requests + Parsel)
for url in office_furniture_urls:
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    
    if response.status_code == 200:
        sel = Selector(text=response.text)
        
        # ✅ Extract product links
        links = sel.css('div.pdp-link a::attr(href)').getall()
        full_links.extend([base_url + link if not link.startswith("http") else link for link in links])
    
    else:
        print(f"Failed to fetch {url}, Status Code: {response.status_code}")

def scrape_product_details(url):
    """Scrape product details from each product page using Selenium."""
    try:
        time.sleep(2)  # Prevent rate limiting
        driver.get(url)
        sel = Selector(text=driver.page_source)  # Convert Selenium page source to Parsel Selector

        # ✅ Extract product details
        name = sel.css('h1.product-name::text').get(default="No name").strip()
        discount = sel.css('span.pricing-sale-percent::text').get(default="No discount").strip()
        price = sel.css('#maincontent span.pricing-default.has-list-price > span >span::text').get(default="No price").strip()

        # ✅ Extract new fields
        material = sel.css('#pdpSummaryMaterials > div > ul > li:nth-child(1) > span::text').get(default="No material").strip()
        designer = sel.css('#pdpSummaryDesigners > div > div > div > div.pdp-summary-designers-bio > p.pdp-summary-designers-name.h7::text').get(default="No designer").strip()
        item_no = sel.css('div.product-peak-wrap span span::text').get(default="No Item No.").strip()
        warranty = sel.css('div#warranty-value-proposition p span::text').get(default="No warranty info").strip()

        return {
            "Name": name,
            "Discount": discount,
            "Price": price,
            "Item No.": item_no,
            "Material": material,
            "Designer": designer,
            "Warranty": warranty,
        }
    
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None

# ✅ Step 2: Scrape product details from all extracted links (Selenium)
all_products = []
for link in full_links:
    product_details = scrape_product_details(link)
    if product_details:
        all_products.append(product_details)

# ✅ Step 3: Convert to DataFrame and save to CSV
office_furniture_df = pd.DataFrame(all_products)
office_furniture_df.to_csv("herman_miller_office_products.csv", index=False)

# Close Selenium WebDriver
driver.quit()
office_furniture_df


,Name,Discount,Price,Item No.,Material,Designer,Warranty
0,Eames Executive Chair,20% off,"$4,420.00",100529336,Tubular steel column,Charles and Ray Eames,12-year warranty
1,Caper Stacking Chair,20% off,$224.00,100100623,Tubular steel frame,Jeff Weber,12-year warranty
2,Eames Soft Pad Chair,20% off,"$3,388.00",8997395,Polished or powder-coated die-cast aluminum fr...,Charles and Ray Eames,12-year warranty
3,Eames Soft Pad Chair,20% off,"$2,828.00",408417,Polished or powder-coated die-cast aluminum fr...,Charles and Ray Eames,12-year warranty
4,Eames Aluminum Group Chair,20% off,"$2,596.00",100317159,Polished or powder-coated die-cast aluminum fr...,Charles and Ray Eames,12-year warranty
...,...,...,...,...,...,...,...
80,Formwork Box,25% off,$45.00,9045051,ABS plastic with nonslip silicone base,Sam Hecht and Kim Colin,1-year warranty
81,Formwork Bin,25% off,$71.25,100181049,ABS plastic with nonslip silicone base,Sam Hecht and Kim Colin,1-year warranty
82,Formwork Pencil Cup,25% off,$15.00,9045149,ABS plastic with nonslip silicone base,Sam Hecht and Kim Colin,1-year warranty
83,Formwork Tray,25% off,$18.75,9044937,ABS plastic with nonslip silicone base,Sam Hecht and Kim Colin,1-year warranty
